# ScaDS.AI Coding: Intolerance only

This notebook annotates the 50 cleaned posts in `Clean_Data_I.csv` for **intolerance toward groups or opposing opinions**.

It uses exactly one system prompt (`systemprompt_i.md`) and writes the model's complete visible response as well as every parsed rating field to a CSV. It does not calculate cognitive simplicity, overconfidence/certainty, or a total extremity score.

Before running, place these three files in the same folder as this notebook:

1. `Clean_Data_I.csv`
2. `systemprompt_i.md`
3. `apikey.txt`


## 1. Imports and settings

The model is fixed to the tested ScaDS.AI model. Fallbacks are disabled so the API cannot silently substitute another model.


In [ ]:
from pathlib import Path
import json
import time

import json_repair
import pandas as pd
from openai import OpenAI

DATA_FILE = Path("Clean_Data_I.csv")
PROMPT_FILE = Path("systemprompt_i.md")
API_KEY_FILE = Path("apikey.txt")

MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"
API_BASE_URL = "https://llm.scads.ai/v1"
TEMPERATURE = 0
REQUEST_PAUSE_SECONDS = 2
MAX_RETRIES = 3


## 2. Load the cleaned sample and initialize the API

The input must contain `id`, `user_name`, and `text`. Duplicate IDs are rejected because every annotation must remain traceable to one post.


In [ ]:
for required_file in (DATA_FILE, PROMPT_FILE, API_KEY_FILE):
    if not required_file.exists():
        raise FileNotFoundError(
            f"Missing {required_file}. Put it in the same folder as this notebook."
        )

d = pd.read_csv(DATA_FILE)
required_columns = {"id", "user_name", "text"}
missing_columns = required_columns.difference(d.columns)
if missing_columns:
    raise ValueError(f"Missing required CSV columns: {sorted(missing_columns)}")

if d["id"].duplicated().any():
    duplicate_ids = d.loc[d["id"].duplicated(keep=False), "id"].tolist()
    raise ValueError(f"Duplicate IDs found: {duplicate_ids}")

if d["text"].isna().any() or d["text"].astype(str).str.strip().eq("").any():
    raise ValueError("Every row must contain non-empty text.")

d = d.loc[:, ["id", "user_name", "text"]].copy()
if len(d) != 50:
    print(f"Warning: expected 50 cleaned posts, but found {len(d)}.")

system_prompt = PROMPT_FILE.read_text(encoding="utf-8").strip()
if not system_prompt:
    raise ValueError(f"The system prompt is empty: {PROMPT_FILE}")

my_api_key = API_KEY_FILE.read_text(encoding="utf-8").strip()
client = OpenAI(base_url=API_BASE_URL, api_key=my_api_key)

print(f"Loaded {len(d)} posts.")
print(f"System prompt: {PROMPT_FILE}")
print(f"Requested model: {MODEL_NAME}")
d.head()


## 3. Annotation function

The parser expects the same JSON fields defined by the existing intolerance system prompt. `llm_reasoning_full` retains the model's entire visible answer, including all of its stated justification, even if JSON parsing fails.


In [ ]:
OUTPUT_FIELDS = [
    "codability",
    "target_assessments",
    "preliminary_score",
    "alternative_score_considered",
    "alternative_score_rejected_because",
    "short_reason",
    "intolerance_score",
]


def extract_json_object(answer: str) -> dict:
    start = answer.find("{")
    end = answer.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("No JSON object found in the model response.")

    parsed = json_repair.loads(answer[start : end + 1])
    if not isinstance(parsed, dict) or not parsed:
        raise ValueError("The parsed model response is not a non-empty JSON object.")
    return parsed


def csv_safe(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def annotate_post(post_text: str) -> dict:
    result = {
        "requested_model": MODEL_NAME,
        "actually_used_model": None,
        **{field: None for field in OUTPUT_FIELDS},
        "llm_reasoning_full": None,
        "error": None,
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": str(post_text)},
                ],
                temperature=TEMPERATURE,
                extra_body={"disable_fallbacks": True},
            )

            answer = response.choices[0].message.content or ""
            result["actually_used_model"] = response.model
            result["llm_reasoning_full"] = answer

            answer_data = extract_json_object(answer)
            for field in OUTPUT_FIELDS:
                result[field] = csv_safe(answer_data.get(field))

            if response.model != MODEL_NAME:
                result["error"] = (
                    f"Model mismatch: requested {MODEL_NAME}, used {response.model}"
                )
            return result

        except Exception as exc:
            result["error"] = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_RETRIES:
                print(
                    f"Attempt {attempt}/{MAX_RETRIES} failed; retrying in 5 seconds: {exc}"
                )
                time.sleep(5)

    return result


## 4. Test one post

Run this cell first. Confirm that `actually_used_model` matches `requested_model`, that the rating fields are populated, and that `error` is empty before processing all 50 posts.


In [ ]:
test_result = annotate_post(d.iloc[0]["text"])

print("Requested model:", test_result["requested_model"])
print("Actually used model:", test_result["actually_used_model"])
print("Error:", test_result["error"])
print("Intolerance score:", test_result["intolerance_score"])
print("Full model response:")
print(test_result["llm_reasoning_full"])


## 5. Annotate all cleaned posts

Each API error is retained in the row instead of stopping the complete run. The two-second pause follows the original notebook's rate-limit precaution.


In [ ]:
results = []

for position, row in enumerate(d.itertuples(index=False), start=1):
    print(f"Processing {position}/{len(d)} | ID: {row.id}...")

    annotation = annotate_post(row.text)
    results.append(
        {
            "id": row.id,
            "user_name": row.user_name,
            "text": row.text,
            **annotation,
        }
    )

    if position < len(d):
        time.sleep(REQUEST_PAUSE_SECONDS)

RESULT_COLUMNS = [
    "id",
    "user_name",
    "text",
    "requested_model",
    "actually_used_model",
    *OUTPUT_FIELDS,
    "llm_reasoning_full",
    "error",
]

d_scored = pd.DataFrame(results, columns=RESULT_COLUMNS)

print(f"Finished {len(d_scored)} posts.")
print(f"Rows with errors: {d_scored['error'].notna().sum()}")
print("Score counts:")
print(d_scored["intolerance_score"].value_counts(dropna=False).sort_index())
d_scored.head()


## 6. Save the LLM annotations

The timestamp prevents accidental overwriting. The semicolon separator and UTF-8 BOM make the result convenient to open in German-language Excel installations.


In [ ]:
timestamp = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = Path(f"intolerance_llm_annotations_{timestamp}.csv")

d_scored.to_csv(
    OUTPUT_FILE,
    index=False,
    sep=";",
    encoding="utf-8-sig",
)

print(f"Saved {len(d_scored)} rows to {OUTPUT_FILE.resolve()}")
